# Chapter 3 - DNN Training, Monte Carlo Simulation and Shapley Attribution
## 60/20/20 Split (Without SamplingForML)

**Dissertation:** Understanding Health Insurance Cost Predictions Using Deep Neural Networks, Monte Carlo Simulation and Shapley Values

This notebook implements the complete Chapter 3 pipeline:
1. Data loading, encoding and 60/20/20 partitioning
2. DNN training with 5-phase hyperparameter comparison
3. Predictive and structural diagnostics
4. Parametric Monte Carlo simulation with joint-distribution validation
5. Residual-augmented MC sensitivity
6. SHAP Shapley value attribution (global + conditional)

## 1. Setup and Imports

In [ ]:
import os, time, copy, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from scipy import stats
import shap

warnings.filterwarnings('ignore')
sns.set_style('white')
plt.rcParams.update({'figure.figsize': (10, 6), 'axes.grid': False, 'font.size': 12,
                      'axes.titlesize': 14, 'axes.labelsize': 12})

FIG_DIR = '/Users/baloyithabangbonganijunior/Downloads/chapter3_figures_60_20_20/'
DATA_DIR = '/Users/baloyithabangbonganijunior/Downloads/'
os.makedirs(FIG_DIR, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print(f'Device: {DEVICE}  |  Seed: {SEED}')

## 2. Load Raw Data and Reference-Category Encoding

In [ ]:
df_raw = pd.read_csv(DATA_DIR + 'insurance_dataset.csv')
df_raw['medical_history'] = df_raw['medical_history'].fillna('None')
df_raw['family_medical_history'] = df_raw['family_medical_history'].fillna('None')
print(f'Raw dataset: {df_raw.shape}')
print(df_raw.dtypes)
print(df_raw.head())

In [ ]:
# Reference-category dummy encoding (same scheme as thesis Table 3.2)
def encode_dataframe(df):
    """Apply reference-category encoding to raw DataFrame, returning 22 features + charges."""
    out = pd.DataFrame()
    out['age'] = df['age'].values.astype(np.float32)
    out['gender'] = (df['gender'] == 'male').astype(np.float32).values
    out['bmi'] = df['bmi'].values.astype(np.float32)
    out['children'] = df['children'].values.astype(np.float32)
    out['smoker'] = (df['smoker'] == 'yes').astype(np.float32).values
    # region (ref = northeast)
    out['region_southwest'] = (df['region'] == 'southwest').astype(np.float32).values
    out['region_northwest'] = (df['region'] == 'northwest').astype(np.float32).values
    out['region_southeast'] = (df['region'] == 'southeast').astype(np.float32).values
    # medical_history (ref = None)
    out['medical_history_Heart_disease'] = (df['medical_history'] == 'Heart disease').astype(np.float32).values
    out['medical_history_High_blood_pressure'] = (df['medical_history'] == 'High blood pressure').astype(np.float32).values
    out['medical_history_Diabetes'] = (df['medical_history'] == 'Diabetes').astype(np.float32).values
    # family_medical_history (ref = None)
    out['family_medical_history_Heart_disease'] = (df['family_medical_history'] == 'Heart disease').astype(np.float32).values
    out['family_medical_history_High_blood_pressure'] = (df['family_medical_history'] == 'High blood pressure').astype(np.float32).values
    out['family_medical_history_Diabetes'] = (df['family_medical_history'] == 'Diabetes').astype(np.float32).values
    # exercise_frequency (ref = Rarely)
    out['exercise_frequency_Occasionally'] = (df['exercise_frequency'] == 'Occasionally').astype(np.float32).values
    out['exercise_frequency_Frequently'] = (df['exercise_frequency'] == 'Frequently').astype(np.float32).values
    out['exercise_frequency_Never'] = (df['exercise_frequency'] == 'Never').astype(np.float32).values
    # occupation (ref = Unemployed)
    out['occupation_Student'] = (df['occupation'] == 'Student').astype(np.float32).values
    out['occupation_Blue_collar'] = (df['occupation'] == 'Blue collar').astype(np.float32).values
    out['occupation_White_collar'] = (df['occupation'] == 'White collar').astype(np.float32).values
    # coverage_level (ref = Basic)
    out['coverage_level_Standard'] = (df['coverage_level'] == 'Standard').astype(np.float32).values
    out['coverage_level_Premium'] = (df['coverage_level'] == 'Premium').astype(np.float32).values
    out['charges'] = df['charges'].values.astype(np.float32)
    return out

df_encoded = encode_dataframe(df_raw)
print(f'Encoded dataset: {df_encoded.shape}')
print(f'Columns ({df_encoded.shape[1]}): {list(df_encoded.columns)}')

## 3. 60/20/20 Train / Validation / Test Split

In [ ]:
# First split: 60% train, 40% temp
df_train, df_temp = train_test_split(df_encoded, test_size=0.40, random_state=SEED)
# Second split: 50% of temp = 20% each for val and test
df_val, df_test = train_test_split(df_temp, test_size=0.50, random_state=SEED)

# Also split the raw DataFrame for MC marginal estimation from training data
df_raw_train = df_raw.iloc[df_train.index].reset_index(drop=True)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

n = len(df_encoded)
print(f'Total     : {n:>10,d}')
print(f'Train     : {len(df_train):>10,d}  ({len(df_train)/n:.2%})')
print(f'Validation: {len(df_val):>10,d}  ({len(df_val)/n:.2%})')
print(f'Test      : {len(df_test):>10,d}  ({len(df_test)/n:.2%})')

In [ ]:
TARGET = 'charges'
FEATURES = [c for c in df_train.columns if c != TARGET]

X_train_raw = df_train[FEATURES].values.astype(np.float32)
y_train_raw = df_train[TARGET].values.astype(np.float32)
X_val_raw = df_val[FEATURES].values.astype(np.float32)
y_val_raw = df_val[TARGET].values.astype(np.float32)
X_test_raw = df_test[FEATURES].values.astype(np.float32)
y_test_raw = df_test[TARGET].values.astype(np.float32)

# Standardisation from training set only
X_mean = X_train_raw.mean(axis=0)
X_std = X_train_raw.std(axis=0)
X_std[X_std == 0] = 1.0

X_train = (X_train_raw - X_mean) / X_std
X_val   = (X_val_raw   - X_mean) / X_std
X_test  = (X_test_raw  - X_mean) / X_std

y_mean = y_train_raw.mean()
y_std  = y_train_raw.std()
y_train = (y_train_raw - y_mean) / y_std
y_val   = (y_val_raw   - y_mean) / y_std
y_test  = (y_test_raw  - y_mean) / y_std

print(f'Features: {len(FEATURES)}')
print(f'y_mean: {y_mean:.2f}, y_std: {y_std:.2f}')

In [ ]:
# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)
y_train_raw_t = torch.tensor(y_train_raw, dtype=torch.float32).unsqueeze(1)
y_val_raw_t = torch.tensor(y_val_raw, dtype=torch.float32).unsqueeze(1)
y_test_raw_t = torch.tensor(y_test_raw, dtype=torch.float32).unsqueeze(1)
print(f'X_train: {X_train_t.shape}, y_train: {y_train_t.shape}')

## 4. DNN Architecture and Helper Functions

In [ ]:
class FunnelDNN(nn.Module):
    """Feed-forward DNN with funnel architecture (bias-free)."""
    def __init__(self, input_dim=22, hidden_layers=[256, 128, 64, 32, 16], activation='CELU'):
        super(FunnelDNN, self).__init__()
        act_map = {'CELU': nn.CELU(), 'GELU': nn.GELU(), 'Tanh': nn.Tanh()}
        act_fn = act_map[activation]
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_layers:
            layers.append(nn.Linear(prev_dim, h_dim, bias=False))
            layers.append(copy.deepcopy(act_fn))
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, 1, bias=False))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

model_check = FunnelDNN(input_dim=22, activation='CELU')
total_params = sum(p.numel() for p in model_check.parameters())
print(f'Architecture: 22 -> 256 -> 128 -> 64 -> 32 -> 16 -> 1')
print(f'Trainable parameters: {total_params:,}')

In [ ]:
def r2_score(y_true, y_pred):
    ss_res = torch.sum((y_true - y_pred) ** 2)
    ss_tot = torch.sum((y_true - y_true.mean()) ** 2)
    return (1 - ss_res / ss_tot).item() if ss_tot != 0 else 0.0

def r2_dot(y_true, y_pred):
    num = torch.dot(y_true.squeeze(), y_pred.squeeze()) ** 2
    den = torch.dot(y_true.squeeze(), y_true.squeeze()) * torch.dot(y_pred.squeeze(), y_pred.squeeze())
    return (num / den).item() if den != 0 else 0.0

def denormalise(y_norm):
    return y_norm * y_std + y_mean

def rmse_fn(y_true, y_pred):
    return torch.sqrt(torch.mean((y_true - y_pred) ** 2)).item()

def mae_fn(y_true, y_pred):
    return torch.mean(torch.abs(y_true - y_pred)).item()

print('Helper functions defined.')

In [ ]:
def train_model(model, X_tr, y_tr, X_va, y_va,
                optimiser_cls, lr, batch_size,
                max_epochs=200, patience=10, val_every=5,
                betas=None, device=DEVICE, verbose=True):
    model = model.to(device)
    criterion = nn.MSELoss()
    if betas is not None:
        optimizer = optimiser_cls(model.parameters(), lr=lr, betas=betas)
    else:
        optimizer = optimiser_cls(model.parameters(), lr=lr)
    train_ds = TensorDataset(X_tr, y_tr)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
    X_va_d, y_va_d = X_va.to(device), y_va.to(device)
    X_tr_d, y_tr_d = X_tr.to(device), y_tr.to(device)
    train_losses, val_epochs, val_losses, train_r2s, val_r2s = [], [], [], [], []
    best_val_r2, best_epoch, best_state, checks_no_improve = -np.inf, 0, None, 0
    start = time.time()
    for epoch in range(1, max_epochs + 1):
        model.train()
        batch_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item() * xb.size(0))
        train_loss = sum(batch_losses) / len(X_tr)
        train_losses.append(train_loss)
        if epoch % val_every == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                preds_tr = model(X_tr_d)
                tr_r2 = r2_dot(y_tr_d, preds_tr)
                preds_va = model(X_va_d)
                val_loss = criterion(preds_va, y_va_d).item()
                va_r2 = r2_dot(y_va_d, preds_va)
            val_epochs.append(epoch)
            val_losses.append(val_loss)
            train_r2s.append(tr_r2)
            val_r2s.append(va_r2)
            if va_r2 > best_val_r2:
                best_val_r2, best_epoch = va_r2, epoch
                best_state = copy.deepcopy(model.state_dict())
                checks_no_improve = 0
            else:
                checks_no_improve += 1
            if verbose:
                print(f'Epoch {epoch:>3d}/{max_epochs} [VAL] | Train Loss: {train_loss:.6f} | '
                      f'Val Loss: {val_loss:.6f} | Train R2: {tr_r2:.6f} | Val R2: {va_r2:.6f}')
            if checks_no_improve >= patience:
                if verbose:
                    print(f'Early stopping at epoch {epoch}. Best epoch: {best_epoch}, Val R2: {best_val_r2:.6f}')
                break
        elif verbose and epoch % 20 == 0:
            print(f'Epoch {epoch:>3d}/{max_epochs}       | Train Loss: {train_loss:.6f}')
    elapsed = time.time() - start
    if verbose:
        print(f'Training complete in {elapsed:.1f}s. Best epoch: {best_epoch}, Val R2: {best_val_r2:.6f}')
    return {'best_model_state': best_state, 'best_epoch': best_epoch, 'best_val_r2': best_val_r2,
            'train_losses': train_losses, 'val_epochs': val_epochs, 'val_losses': val_losses,
            'train_r2s': train_r2s, 'val_r2s': val_r2s, 'elapsed': elapsed}

print('Training function defined.')

## 5. Hyperparameter Comparison (5 Phases)

In [ ]:
MAX_EPOCHS = 100
PATIENCE = 10
VAL_EVERY = 5
comparison_results = {}

# Phase 1: Activation
print('=' * 70)
print('PHASE 1: ACTIVATION FUNCTION (fixed: lr=0.001, bs=256, Adam)')
print('=' * 70)
phase1_results = {}
for act_name in ['CELU', 'GELU', 'Tanh']:
    print(f'\n--- {act_name} ---')
    torch.manual_seed(SEED)
    model = FunnelDNN(input_dim=22, activation=act_name)
    result = train_model(model, X_train_t, y_train_t, X_val_t, y_val_t,
                         optimiser_cls=optim.Adam, lr=0.001, batch_size=256,
                         max_epochs=MAX_EPOCHS, patience=PATIENCE, val_every=VAL_EVERY)
    phase1_results[act_name] = result
comparison_results['Phase 1'] = phase1_results
best_activation = max(phase1_results, key=lambda k: phase1_results[k]['best_val_r2'])
print(f'\n>>> Best activation: {best_activation} (Val R2 = {phase1_results[best_activation]["best_val_r2"]:.6f})')

In [ ]:
# Phase 2: Learning Rate
print('=' * 70)
print(f'PHASE 2: LEARNING RATE (fixed: {best_activation}, bs=256, Adam)')
print('=' * 70)
phase2_results = {}
for lr_val in [0.01, 0.001, 0.0005, 0.0001]:
    lr_label = str(lr_val)
    print(f'\n--- lr={lr_val} ---')
    torch.manual_seed(SEED)
    model = FunnelDNN(input_dim=22, activation=best_activation)
    result = train_model(model, X_train_t, y_train_t, X_val_t, y_val_t,
                         optimiser_cls=optim.Adam, lr=lr_val, batch_size=256,
                         max_epochs=MAX_EPOCHS, patience=PATIENCE, val_every=VAL_EVERY)
    phase2_results[lr_label] = result
comparison_results['Phase 2'] = phase2_results
best_lr_label = max(phase2_results, key=lambda k: phase2_results[k]['best_val_r2'])
best_lr = float(best_lr_label)
print(f'\n>>> Best lr: {best_lr} (Val R2 = {phase2_results[best_lr_label]["best_val_r2"]:.6f})')

In [ ]:
# Phase 3: Batch Size
print('=' * 70)
print(f'PHASE 3: BATCH SIZE (fixed: {best_activation}, lr={best_lr}, Adam)')
print('=' * 70)
phase3_results = {}
for bs in [64, 128, 256, 512]:
    bs_label = str(bs)
    print(f'\n--- bs={bs} ---')
    torch.manual_seed(SEED)
    model = FunnelDNN(input_dim=22, activation=best_activation)
    result = train_model(model, X_train_t, y_train_t, X_val_t, y_val_t,
                         optimiser_cls=optim.Adam, lr=best_lr, batch_size=bs,
                         max_epochs=MAX_EPOCHS, patience=PATIENCE, val_every=VAL_EVERY)
    phase3_results[bs_label] = result
comparison_results['Phase 3'] = phase3_results
best_bs_label = max(phase3_results, key=lambda k: phase3_results[k]['best_val_r2'])
best_bs = int(best_bs_label)
print(f'\n>>> Best batch size: {best_bs} (Val R2 = {phase3_results[best_bs_label]["best_val_r2"]:.6f})')

In [ ]:
# Phase 4: Optimiser
print('=' * 70)
print(f'PHASE 4: OPTIMISER (fixed: {best_activation}, lr={best_lr}, bs={best_bs})')
print('=' * 70)
optimiser_candidates = {'Adam': optim.Adam, 'NAdam': optim.NAdam}
phase4_results = {}
for opt_name, opt_cls in optimiser_candidates.items():
    print(f'\n--- {opt_name} ---')
    torch.manual_seed(SEED)
    model = FunnelDNN(input_dim=22, activation=best_activation)
    result = train_model(model, X_train_t, y_train_t, X_val_t, y_val_t,
                         optimiser_cls=opt_cls, lr=best_lr, batch_size=best_bs,
                         max_epochs=MAX_EPOCHS, patience=PATIENCE, val_every=VAL_EVERY)
    phase4_results[opt_name] = result
comparison_results['Phase 4'] = phase4_results
best_optimiser_name = max(phase4_results, key=lambda k: phase4_results[k]['best_val_r2'])
best_optimiser_cls = optimiser_candidates[best_optimiser_name]
print(f'\n>>> Best optimiser: {best_optimiser_name} (Val R2 = {phase4_results[best_optimiser_name]["best_val_r2"]:.6f})')

In [ ]:
# Phase 5: Momentum
print('=' * 70)
print(f'PHASE 5: MOMENTUM (fixed: {best_activation}, lr={best_lr}, bs={best_bs}, {best_optimiser_name})')
print('=' * 70)
momentum_candidates = {
    'B1=0.90': (0.90, 0.999),
    'B1=0.95': (0.95, 0.999),
    'B1=0.99': (0.99, 0.999),
}
phase5_results = {}
for mom_label, betas_val in momentum_candidates.items():
    print(f'\n--- {mom_label} ---')
    torch.manual_seed(SEED)
    model = FunnelDNN(input_dim=22, activation=best_activation)
    result = train_model(model, X_train_t, y_train_t, X_val_t, y_val_t,
                         optimiser_cls=best_optimiser_cls, lr=best_lr, batch_size=best_bs,
                         max_epochs=MAX_EPOCHS, patience=PATIENCE, val_every=VAL_EVERY,
                         betas=betas_val)
    phase5_results[mom_label] = result
comparison_results['Phase 5'] = phase5_results
best_momentum_label = max(phase5_results, key=lambda k: phase5_results[k]['best_val_r2'])
best_betas = momentum_candidates[best_momentum_label]
print(f'\n>>> Best momentum: {best_momentum_label}, betas={best_betas}')
print(f'    Val R2 = {phase5_results[best_momentum_label]["best_val_r2"]:.6f}')

## 6. Hyperparameter Comparison Summary and Figures

In [ ]:
# Grand summary
print('=' * 70)
print('HYPERPARAMETER COMPARISON SUMMARY')
print('=' * 70)
print(f'Activation : {best_activation}')
print(f'Learning rate: {best_lr}')
print(f'Batch size : {best_bs}')
print(f'Optimiser  : {best_optimiser_name}')
print(f'Momentum   : B1={best_betas[0]}, B2={best_betas[1]}')

rows = []
for phase_name, phase_res in comparison_results.items():
    for config_name, res in phase_res.items():
        rows.append({'Phase': phase_name, 'Configuration': config_name,
                     'Best Epoch': res['best_epoch'], 'Val R2': round(res['best_val_r2'], 6),
                     'Time (s)': round(res['elapsed'], 1)})
df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))
df_summary.to_csv(FIG_DIR + 'hyperparameter_comparison.csv', index=False)

In [ ]:
# Phase comparison figures
phase_data = [
    ('Phase 1: Activation', phase1_results, None),
    ('Phase 2: Learning Rate', phase2_results, 'lr='),
    ('Phase 3: Batch Size', phase3_results, 'bs='),
    ('Phase 4: Optimiser', phase4_results, None),
    ('Phase 5: Momentum', phase5_results, None),
]
for title, results, prefix in phase_data:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for label, res in results.items():
        display_label = f'{prefix}{label}' if prefix else label
        axes[0].plot(res['val_epochs'], res['val_losses'], label=display_label, marker='o', markersize=3)
        axes[1].plot(res['val_epochs'], res['val_r2s'], label=display_label, marker='o', markersize=3)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Validation Loss (MSE)')
    axes[0].set_title(f'{title}: Validation Loss'); axes[0].legend(); sns.despine(ax=axes[0])
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Validation R2')
    axes[1].set_title(f'{title}: Validation R2'); axes[1].legend(); sns.despine(ax=axes[1])
    plt.tight_layout()
    safe_title = title.replace(' ', '_').replace(':', '').lower()
    plt.savefig(FIG_DIR + f'fig_{safe_title}_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved: fig_{safe_title}_comparison.png')

## 7. Final Model Training

In [ ]:
print('=' * 70)
print('FINAL MODEL TRAINING')
print(f'Architecture : 22 -> 256 -> 128 -> 64 -> 32 -> 16 -> 1')
print(f'Config: {best_activation}, lr={best_lr}, bs={best_bs}, {best_optimiser_name}, betas={best_betas}')
print('=' * 70)

torch.manual_seed(SEED)
final_model = FunnelDNN(input_dim=22, activation=best_activation)
final_result = train_model(
    final_model, X_train_t, y_train_t, X_val_t, y_val_t,
    optimiser_cls=best_optimiser_cls, lr=best_lr, batch_size=best_bs,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, val_every=VAL_EVERY,
    betas=best_betas)

final_model.load_state_dict(final_result['best_model_state'])
final_model = final_model.to(DEVICE)
final_model.eval()
print(f'Final model loaded from epoch {final_result["best_epoch"]}')

## 8. Model Evaluation and Diagnostics

In [ ]:
# Evaluate on all sets (de-normalised scale)
final_model.eval()
with torch.no_grad():
    preds_train_raw = denormalise(final_model(X_train_t.to(DEVICE)).cpu())
    preds_val_raw = denormalise(final_model(X_val_t.to(DEVICE)).cpu())
    preds_test_raw = denormalise(final_model(X_test_t.to(DEVICE)).cpu())

metrics = {}
for set_name, y_true, y_pred in [
    ('Train', y_train_raw_t, preds_train_raw),
    ('Validation', y_val_raw_t, preds_val_raw),
    ('Test', y_test_raw_t, preds_test_raw)]:
    metrics[set_name] = {'R2': r2_score(y_true, y_pred), 'RMSE': rmse_fn(y_true, y_pred),
                         'MAE': mae_fn(y_true, y_pred)}

print('=' * 60)
print('FINAL MODEL EVALUATION (De-normalised, Rand scale)')
print('=' * 60)
print(f'{"Set":<12} {"R2":>10} {"RMSE (R)":>12} {"MAE (R)":>12}')
print('-' * 46)
for s, m in metrics.items():
    print(f'{s:<12} {m["R2"]:>10.6f} {m["RMSE"]:>12.2f} {m["MAE"]:>12.2f}')

In [ ]:
# Training and validation curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
all_epochs = range(1, len(final_result['train_losses']) + 1)
axes[0].plot(all_epochs, final_result['train_losses'], label='Training Loss', color='#2196F3', alpha=0.7)
axes[0].plot(final_result['val_epochs'], final_result['val_losses'], label='Validation Loss', color='#F44336', marker='o', markersize=4)
axes[0].axvline(x=final_result['best_epoch'], color='green', linestyle='--', alpha=0.7, label=f'Best epoch (e*={final_result["best_epoch"]})')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training and Validation Loss'); axes[0].legend(); sns.despine(ax=axes[0])
axes[1].plot(final_result['val_epochs'], final_result['train_r2s'], label='Training R2', color='#2196F3', marker='o', markersize=4)
axes[1].plot(final_result['val_epochs'], final_result['val_r2s'], label='Validation R2', color='#F44336', marker='o', markersize=4)
axes[1].axvline(x=final_result['best_epoch'], color='green', linestyle='--', alpha=0.7, label=f'Best epoch (e*={final_result["best_epoch"]})')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('R2')
axes[1].set_title('Training and Validation R2'); axes[1].legend(); sns.despine(ax=axes[1])
plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_training_validation_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Training and validation losses over epochs
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(all_epochs, final_result['train_losses'], label='Train Loss', color='tab:blue', linewidth=1.5)
ax.plot(final_result['val_epochs'], final_result['val_losses'], label='Validation Loss', color='tab:orange', linewidth=1.5)
ax.set_xlabel('Epochs'); ax.set_ylabel('Loss'); ax.set_title('Training and Validation Losses Over Epochs')
ax.legend(); sns.despine(ax=ax)
plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_train_val_losses_epochs.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Actual vs Predicted (sorted) - Train and Test
for set_label, y_act_arr, y_prd_arr, r2_val in [
    ('Train', y_train_raw_t.numpy().flatten(), preds_train_raw.numpy().flatten(), metrics['Train']['R2']),
    ('Test', y_test_raw_t.numpy().flatten(), preds_test_raw.numpy().flatten(), metrics['Test']['R2'])]:
    sort_idx = np.argsort(y_act_arr)
    step = max(1, len(y_act_arr) // 10000)
    plot_idx = np.arange(0, len(y_act_arr), step)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(plot_idx, y_act_arr[sort_idx][plot_idx], color='tab:blue', linewidth=1.2, label='Actual', alpha=0.8)
    ax.plot(plot_idx, y_prd_arr[sort_idx][plot_idx], color='tab:orange', linewidth=1.2, label='Predicted', alpha=0.8)
    ax.set_xlabel('Observation Index (sorted)'); ax.set_ylabel('Charges (R)')
    ax.set_title(f'Actual vs Predicted - {set_label} Set (R2 = {r2_val:.4f})')
    ax.legend(); sns.despine(ax=ax); plt.tight_layout()
    plt.savefig(FIG_DIR + f'fig_actual_vs_predicted_sorted_{set_label.lower()}.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Violin plots: Actual vs Predicted
y_act_train_np = y_train_raw_t.numpy().flatten()
y_prd_train_np = preds_train_raw.numpy().flatten()
y_act_test_np = y_test_raw_t.numpy().flatten()
y_prd_test_np = preds_test_raw.numpy().flatten()
rng_v = np.random.default_rng(SEED)
n_v = min(50000, len(y_act_train_np))
vi_tr = rng_v.choice(len(y_act_train_np), n_v, replace=False)
vi_te = rng_v.choice(len(y_act_test_np), min(n_v, len(y_act_test_np)), replace=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, vi, y_a, y_p, label, r2v in [
    (axes[0], vi_tr, y_act_train_np, y_prd_train_np, 'Train', metrics['Train']['R2']),
    (axes[1], vi_te, y_act_test_np, y_prd_test_np, 'Test', metrics['Test']['R2'])]:
    df_v = pd.DataFrame({'Charges (R)': np.concatenate([y_a[vi], y_p[vi]]),
                         'Type': ['Actual']*len(vi) + ['Predicted']*len(vi)})
    sns.violinplot(data=df_v, x='Type', y='Charges (R)', ax=ax,
                   palette=['tab:blue', 'tab:orange'], inner='quartile', cut=0)
    ax.set_title(f'{label} Set (R2 = {r2v:.4f})'); ax.set_xlabel(''); sns.despine(ax=ax)
plt.suptitle('Distribution of Actual vs Predicted Charges', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_violin_actual_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.show()

# Violin: residuals
res_tr = y_act_train_np - y_prd_train_np
res_te = y_act_test_np - y_prd_test_np
df_vr = pd.DataFrame({'Residual (R)': np.concatenate([res_tr[vi_tr], res_te[vi_te]]),
                       'Set': ['Train']*len(vi_tr) + ['Test']*len(vi_te)})
fig, ax = plt.subplots(figsize=(8, 6))
sns.violinplot(data=df_vr, x='Set', y='Residual (R)', ax=ax,
               palette=['tab:blue', 'tab:orange'], inner='quartile', cut=0)
ax.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.7)
ax.set_title('Residual Distributions'); ax.set_xlabel(''); sns.despine(ax=ax)
plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_violin_residuals.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Residual diagnostics (test set)
residuals_test = (y_test_raw_t - preds_test_raw).numpy().flatten()
fitted_test = preds_test_raw.numpy().flatten()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
n_plot = min(10000, len(residuals_test))
idx = np.random.choice(len(residuals_test), n_plot, replace=False)

axes[0,0].scatter(fitted_test[idx], residuals_test[idx], alpha=0.3, s=5, color='#2196F3')
axes[0,0].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[0,0].set_xlabel('Fitted Values (R)'); axes[0,0].set_ylabel('Residuals (R)')
axes[0,0].set_title('(a) Residuals vs Fitted'); sns.despine(ax=axes[0,0])

axes[0,1].hist(residuals_test, bins=80, color='#2196F3', edgecolor='white', alpha=0.8)
axes[0,1].axvline(x=0, color='red', linestyle='--', linewidth=1)
axes[0,1].set_xlabel('Residual (R)'); axes[0,1].set_ylabel('Frequency')
axes[0,1].set_title('(b) Residual Distribution'); sns.despine(ax=axes[0,1])

std_res = (residuals_test - residuals_test.mean()) / residuals_test.std()
qq_sample = np.sort(np.random.choice(std_res, min(5000, len(std_res)), replace=False))
theoretical = stats.norm.ppf(np.linspace(0.001, 0.999, len(qq_sample)))
axes[1,0].scatter(theoretical, qq_sample, alpha=0.3, s=5, color='#2196F3')
axes[1,0].plot([-4,4], [-4,4], 'r--', linewidth=1)
axes[1,0].set_xlabel('Theoretical Quantiles'); axes[1,0].set_ylabel('Sample Quantiles')
axes[1,0].set_title('(c) Q-Q Plot'); sns.despine(ax=axes[1,0])

idx_seq = np.sort(np.random.choice(len(residuals_test), n_plot, replace=False))
axes[1,1].scatter(idx_seq, residuals_test[idx_seq], alpha=0.3, s=5, color='#2196F3')
axes[1,1].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[1,1].set_xlabel('Observation Index'); axes[1,1].set_ylabel('Residual (R)')
axes[1,1].set_title('(d) Residuals vs Order'); sns.despine(ax=axes[1,1])

plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_residual_diagnostics.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Residual stats (test): mean={residuals_test.mean():.2f}, SD={residuals_test.std():.2f}, '
      f'skew={stats.skew(residuals_test):.4f}, kurt={stats.kurtosis(residuals_test):.4f}')

In [ ]:
# PCA structural diagnostics
pca = PCA(n_components=2, random_state=SEED)
pca.fit(X_train)
pc_train = pca.transform(X_train)
pc_test = pca.transform(X_test)
ev1 = pca.explained_variance_ratio_[0] * 100
ev2 = pca.explained_variance_ratio_[1] * 100
print(f'PC1: {ev1:.1f}%, PC2: {ev2:.1f}%, Cumulative: {ev1+ev2:.1f}%')

vmin = min(y_act_train_np.min(), y_act_test_np.min(), y_prd_train_np.min(), y_prd_test_np.min())
vmax = max(y_act_train_np.max(), y_act_test_np.max(), y_prd_train_np.max(), y_prd_test_np.max())

for pc, colour_vals, title, fname in [
    (pc_train, y_act_train_np, 'Train PCA: Actual Target', 'fig_sml_pca_train_actual.png'),
    (pc_train, y_prd_train_np, 'Train PCA: Predicted Value', 'fig_sml_pca_train_predicted.png'),
    (pc_test, y_act_test_np, 'Test PCA: Actual Target', 'fig_sml_pca_test_actual.png'),
    (pc_test, y_prd_test_np, 'Test PCA: Predicted Value', 'fig_sml_pca_test_predicted.png')]:
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(pc[:,0], pc[:,1], c=colour_vals, cmap='viridis', s=6, alpha=0.5,
                    vmin=vmin, vmax=vmax, edgecolors='none')
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_xlabel(f'PC1 ({ev1:.1f}%)'); ax.set_ylabel(f'PC2 ({ev2:.1f}%)')
    ax.set_title(title); plt.colorbar(sc, ax=ax); sns.despine(ax=ax)
    plt.tight_layout()
    plt.savefig(FIG_DIR + fname, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# PCA loadings
loadings = pd.DataFrame(pca.components_.T, index=FEATURES, columns=['PC1', 'PC2'])
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for i, pc_col in enumerate(['PC1', 'PC2']):
    ld = loadings[[pc_col]].copy()
    ld['abs'] = ld[pc_col].abs()
    ld = ld.sort_values('abs', ascending=True)
    colours = ['#2196F3' if v >= 0 else '#F44336' for v in ld[pc_col]]
    axes[i].barh(ld.index, ld[pc_col], color=colours, edgecolor='white')
    axes[i].axvline(x=0, color='black', linewidth=0.8)
    axes[i].set_xlabel('Loading'); axes[i].set_title(f'{pc_col} Loadings')
    sns.despine(ax=axes[i])
plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_pca_loadings.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmaps
def plot_heatmap(corr, title, fname):
    fig, ax = plt.subplots(figsize=(14, 12))
    im = ax.imshow(corr.values, cmap='viridis', vmin=-1, vmax=1, aspect='auto')
    n = len(corr)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(corr.index, fontsize=9)
    plt.colorbar(im, ax=ax, label='Correlation')
    ax.set_title(title, fontsize=14); plt.tight_layout()
    plt.savefig(FIG_DIR + fname, dpi=300, bbox_inches='tight')
    plt.show()

for X_arr, y_a, y_p, label in [
    (X_train, y_act_train_np, y_prd_train_np, 'train'),
    (X_test, y_act_test_np, y_prd_test_np, 'test')]:
    df_a = pd.DataFrame(X_arr, columns=FEATURES); df_a['actual_target'] = y_a
    df_p = pd.DataFrame(X_arr, columns=FEATURES); df_p['predicted_target'] = y_p
    plot_heatmap(df_a.corr(), f'{label.title()} Heatmap: Actual Target', f'fig_sml_heatmap_{label}_actual.png')
    plot_heatmap(df_p.corr(), f'{label.title()} Heatmap: Predicted Target', f'fig_sml_heatmap_{label}_predicted.png')

In [ ]:
# Save model checkpoint
model_path = FIG_DIR + 'chapter3_final_model_60_20_20.pth'
checkpoint = {
    'model_state_dict': final_model.state_dict(),
    'architecture': {'input_dim': 22, 'hidden_layers': [256,128,64,32,16],
                     'activation': best_activation, 'bias': False},
    'hyperparameters': {'learning_rate': best_lr, 'batch_size': best_bs,
                        'optimiser': best_optimiser_name, 'betas': best_betas,
                        'max_epochs': MAX_EPOCHS, 'val_every': VAL_EVERY,
                        'patience': PATIENCE, 'seed': SEED,
                        'best_epoch': final_result['best_epoch']},
    'standardisation': {'X_mean': X_mean, 'X_std': X_std,
                        'y_mean': float(y_mean), 'y_std': float(y_std)},
    'features': FEATURES,
    'metrics': metrics,
    'split': {'train': len(df_train), 'val': len(df_val), 'test': len(df_test)}
}
torch.save(checkpoint, model_path)
print(f'Model saved: {model_path}')

## 9. Parametric Monte Carlo Simulation

In [ ]:
def score_dnn(X_encoded_raw, standardise=True, denorm=True):
    X = X_encoded_raw.astype(np.float32)
    if standardise:
        X = (X - X_mean) / X_std
    X_t = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        preds = final_model(X_t).cpu().numpy().flatten()
    if denorm:
        preds = preds * y_std + y_mean
    return preds

# Estimate marginal parameters from TRAINING data only
p_male = (df_raw_train['gender'] == 'male').mean()
p_smoker = (df_raw_train['smoker'] == 'yes').mean()

region_levels = ['northeast', 'northwest', 'southeast', 'southwest']
region_probs = np.array([(df_raw_train['region'] == r).mean() for r in region_levels])
med_hist_levels = ['None', 'Heart disease', 'High blood pressure', 'Diabetes']
med_hist_probs = np.array([(df_raw_train['medical_history'] == m).mean() for m in med_hist_levels])
fam_hist_levels = ['None', 'Heart disease', 'High blood pressure', 'Diabetes']
fam_hist_probs = np.array([(df_raw_train['family_medical_history'] == f).mean() for f in fam_hist_levels])
exercise_levels = ['Rarely', 'Occasionally', 'Frequently', 'Never']
exercise_probs = np.array([(df_raw_train['exercise_frequency'] == e).mean() for e in exercise_levels])
occ_levels = ['Unemployed', 'Student', 'Blue collar', 'White collar']
occ_probs = np.array([(df_raw_train['occupation'] == o).mean() for o in occ_levels])
cov_levels = ['Basic', 'Standard', 'Premium']
cov_probs = np.array([(df_raw_train['coverage_level'] == c).mean() for c in cov_levels])

print('Marginal parameters (from training set):')
print(f'  p_male={p_male:.4f}, p_smoker={p_smoker:.4f}')

In [ ]:
B = 10_000

def generate_mc_profiles(n, rng):
    return pd.DataFrame({
        'age': rng.integers(18, 66, size=n),
        'gender': rng.choice(['male','female'], size=n, p=[p_male, 1-p_male]),
        'bmi': rng.uniform(18.0, 50.0, size=n),
        'children': rng.integers(0, 6, size=n),
        'smoker': rng.choice(['yes','no'], size=n, p=[p_smoker, 1-p_smoker]),
        'region': rng.choice(region_levels, size=n, p=region_probs),
        'medical_history': rng.choice(med_hist_levels, size=n, p=med_hist_probs),
        'family_medical_history': rng.choice(fam_hist_levels, size=n, p=fam_hist_probs),
        'exercise_frequency': rng.choice(exercise_levels, size=n, p=exercise_probs),
        'occupation': rng.choice(occ_levels, size=n, p=occ_probs),
        'coverage_level': rng.choice(cov_levels, size=n, p=cov_probs)
    })

def encode_profiles(df_mc):
    n = len(df_mc)
    X = np.zeros((n, 22), dtype=np.float32)
    X[:,0] = df_mc['age'].values
    X[:,1] = (df_mc['gender']=='male').astype(np.float32).values
    X[:,2] = df_mc['bmi'].values
    X[:,3] = df_mc['children'].values
    X[:,4] = (df_mc['smoker']=='yes').astype(np.float32).values
    X[:,5] = (df_mc['region']=='southwest').astype(np.float32).values
    X[:,6] = (df_mc['region']=='northwest').astype(np.float32).values
    X[:,7] = (df_mc['region']=='southeast').astype(np.float32).values
    X[:,8] = (df_mc['medical_history']=='Heart disease').astype(np.float32).values
    X[:,9] = (df_mc['medical_history']=='High blood pressure').astype(np.float32).values
    X[:,10] = (df_mc['medical_history']=='Diabetes').astype(np.float32).values
    X[:,11] = (df_mc['family_medical_history']=='Heart disease').astype(np.float32).values
    X[:,12] = (df_mc['family_medical_history']=='High blood pressure').astype(np.float32).values
    X[:,13] = (df_mc['family_medical_history']=='Diabetes').astype(np.float32).values
    X[:,14] = (df_mc['exercise_frequency']=='Occasionally').astype(np.float32).values
    X[:,15] = (df_mc['exercise_frequency']=='Frequently').astype(np.float32).values
    X[:,16] = (df_mc['exercise_frequency']=='Never').astype(np.float32).values
    X[:,17] = (df_mc['occupation']=='Student').astype(np.float32).values
    X[:,18] = (df_mc['occupation']=='Blue collar').astype(np.float32).values
    X[:,19] = (df_mc['occupation']=='White collar').astype(np.float32).values
    X[:,20] = (df_mc['coverage_level']=='Standard').astype(np.float32).values
    X[:,21] = (df_mc['coverage_level']=='Premium').astype(np.float32).values
    return X

rng_mc = np.random.default_rng(SEED)
df_mc = generate_mc_profiles(B, rng_mc)
X_mc_encoded = encode_profiles(df_mc)
y_mc = score_dnn(X_mc_encoded)

print(f'MC profiles: {B}')
print(f'Predicted cost range: R{y_mc.min():,.2f} to R{y_mc.max():,.2f}')
print(f'Mean: R{y_mc.mean():,.2f}, Median: R{np.median(y_mc):,.2f}')

In [ ]:
# MC convergence diagnostics
pilot_sizes = [1000, 2500, 5000, 10000, 20000]
rng_pilot = np.random.default_rng(SEED)
df_pilot = generate_mc_profiles(max(pilot_sizes), rng_pilot)
X_pilot_enc = encode_profiles(df_pilot)
y_pilot = score_dnn(X_pilot_enc)

convergence_stats = []
for B_p in pilot_sizes:
    y_sub = y_pilot[:B_p]
    convergence_stats.append({'B': B_p, 'Mean': y_sub.mean(), 'SD': y_sub.std(),
        'P50': np.percentile(y_sub, 50), 'P90': np.percentile(y_sub, 90),
        'P95': np.percentile(y_sub, 95), 'P99': np.percentile(y_sub, 99),
        'MCSE': y_sub.std() / np.sqrt(B_p)})
df_conv = pd.DataFrame(convergence_stats)
print(df_conv.to_string(index=False, float_format='%.2f'))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for idx, stat in enumerate(['Mean', 'SD', 'P50', 'P90', 'P95', 'P99']):
    ax = axes[idx//3, idx%3]
    ax.plot(df_conv['B'], df_conv[stat], marker='o', color='#2196F3', linewidth=1.5)
    ax.set_xlabel('Sample Size (B)'); ax.set_ylabel(f'{stat} (R)'); ax.set_title(f'Running {stat}')
    ref = df_conv.loc[df_conv['B']==10000, stat].values[0]
    ax.axhline(y=ref, color='red', linestyle='--', alpha=0.5, linewidth=0.8); sns.despine(ax=ax)
plt.suptitle('MC Convergence Diagnostics', fontsize=14, y=1.01); plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_mc_convergence.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# MC predicted cost distribution
mc_summary = {'Mean': y_mc.mean(), 'SD': y_mc.std(), 'Skewness': float(stats.skew(y_mc)),
              'Kurtosis': float(stats.kurtosis(y_mc)), 'Min': y_mc.min(),
              'P25': np.percentile(y_mc, 25), 'Median': np.percentile(y_mc, 50),
              'P75': np.percentile(y_mc, 75), 'P90': np.percentile(y_mc, 90),
              'P95': np.percentile(y_mc, 95), 'P99': np.percentile(y_mc, 99), 'Max': y_mc.max()}
print('MC Predicted-Cost Summary:')
for k, v in mc_summary.items(): print(f'  {k:<12}: {v:>12.2f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(y_mc, bins=80, color='#2196F3', edgecolor='white', alpha=0.7, density=True)
kde_x = np.linspace(y_mc.min(), y_mc.max(), 500)
axes[0].plot(kde_x, stats.gaussian_kde(y_mc)(kde_x), color='#F44336', linewidth=1.5)
axes[0].axvline(y_mc.mean(), color='black', linestyle='--', linewidth=1, label=f'Mean=R{y_mc.mean():,.0f}')
axes[0].set_xlabel('Predicted Cost (R)'); axes[0].set_ylabel('Density')
axes[0].set_title('MC Predicted-Cost Distribution'); axes[0].legend(); sns.despine(ax=axes[0])
axes[1].boxplot(y_mc, vert=True, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor='#2196F3', alpha=0.5), medianprops=dict(color='red', linewidth=1.5))
axes[1].set_ylabel('Predicted Cost (R)'); axes[1].set_title('Box Plot'); axes[1].set_xticklabels(['MC'])
sns.despine(ax=axes[1]); plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_mc_predicted_cost_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Joint-Distribution Validation

The product-marginal MC design assumes independence among predictors. This section formally
compares the joint distribution of the simulated profiles against the training data to diagnose
any discrepancy introduced by the independence assumption.

In [ ]:
# Pearson correlation comparison (numerical variables)
num_cols = ['age', 'bmi', 'children']
corr_train_num = df_raw_train[num_cols].corr()
corr_mc_num = df_mc[num_cols].corr()

print('Pearson Correlation Comparison (numerical variables):')
print('\nTraining data:')
print(corr_train_num.round(4).to_string())
print('\nMC simulated:')
print(corr_mc_num.round(4).to_string())
print('\nAbsolute difference:')
print((corr_train_num - corr_mc_num).abs().round(4).to_string())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, corr, title in [(axes[0], corr_train_num, 'Training'), (axes[1], corr_mc_num, 'MC Simulated'),
                         (axes[2], (corr_train_num - corr_mc_num).abs(), 'Absolute Difference')]:
    im = ax.imshow(corr.values, cmap='viridis', vmin=-1 if title != 'Absolute Difference' else 0,
                   vmax=1 if title != 'Absolute Difference' else 0.1, aspect='auto')
    ax.set_xticks(range(len(corr))); ax.set_yticks(range(len(corr)))
    ax.set_xticklabels(corr.columns, rotation=45, ha='right')
    ax.set_yticklabels(corr.index)
    for i in range(len(corr)):
        for j in range(len(corr)):
            ax.text(j, i, f'{corr.values[i,j]:.3f}', ha='center', va='center', fontsize=10)
    plt.colorbar(im, ax=ax); ax.set_title(title); sns.despine(ax=ax)
plt.suptitle('Joint Correlation Comparison: Training vs MC', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_mc_joint_correlation_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cramer's V for categorical associations
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(ct)[0]
    n = ct.sum().sum()
    r, k = ct.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1))) if min(r, k) > 1 else 0.0

cat_cols = ['gender', 'smoker', 'region', 'medical_history',
            'family_medical_history', 'exercise_frequency', 'occupation', 'coverage_level']

cv_train = pd.DataFrame(np.zeros((len(cat_cols), len(cat_cols))), index=cat_cols, columns=cat_cols)
cv_mc = pd.DataFrame(np.zeros((len(cat_cols), len(cat_cols))), index=cat_cols, columns=cat_cols)

for i, c1 in enumerate(cat_cols):
    for j, c2 in enumerate(cat_cols):
        if i <= j:
            v_tr = cramers_v(df_raw_train[c1], df_raw_train[c2])
            v_mc = cramers_v(df_mc[c1], df_mc[c2])
            cv_train.iloc[i, j] = cv_train.iloc[j, i] = v_tr
            cv_mc.iloc[i, j] = cv_mc.iloc[j, i] = v_mc

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, cv, title in [(axes[0], cv_train, 'Training'), (axes[1], cv_mc, 'MC Simulated')]:
    im = ax.imshow(cv.values, cmap='viridis', vmin=0, vmax=0.1, aspect='auto')
    ax.set_xticks(range(len(cat_cols))); ax.set_yticks(range(len(cat_cols)))
    ax.set_xticklabels(cat_cols, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(cat_cols, fontsize=8)
    for ii in range(len(cat_cols)):
        for jj in range(len(cat_cols)):
            ax.text(jj, ii, f'{cv.values[ii,jj]:.3f}', ha='center', va='center', fontsize=7)
    plt.colorbar(im, ax=ax); ax.set_title(title)
plt.suptitle("Cramer's V: Training vs MC Simulated", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_mc_joint_cramers_v.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# PCA comparison: training vs MC encoded inputs
X_mc_std = ((X_mc_encoded - X_mean) / X_std).astype(np.float32)
pca_joint = PCA(n_components=2, random_state=SEED)
pca_joint.fit(X_train)
pc_train_j = pca_joint.transform(X_train[:10000])
pc_mc_j = pca_joint.transform(X_mc_std)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(pc_train_j[:,0], pc_train_j[:,1], alpha=0.3, s=5, color='#2196F3')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].set_title('Training Data (first 10k)'); axes[0].set_xlim(-3,3); axes[0].set_ylim(-3,3)
sns.despine(ax=axes[0])
axes[1].scatter(pc_mc_j[:,0], pc_mc_j[:,1], alpha=0.3, s=5, color='#F44336')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].set_title('MC Simulated (10k)'); axes[1].set_xlim(-3,3); axes[1].set_ylim(-3,3)
sns.despine(ax=axes[1])
plt.suptitle('PCA Comparison: Training vs MC Input Space', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_mc_joint_pca_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 10b. SHAP Bootstrap vs Independent MC Invariance Test

Following supervisor feedback: the critical validation is not whether the individual marginals match,
but whether the **joint distribution** affects the Shapley attribution. This test compares SHAP
values computed on (a) the independent parametric MC sample and (b) a bootstrap MC sample that
resamples full rows from training data, preserving the empirical joint dependence structure.
If the rankings and magnitudes are similar, the independence assumption does not distort the
Shapley results.

In [ ]:
# Bootstrap MC: resample full rows from training data (preserves joint distribution)
rng_boot = np.random.default_rng(SEED + 99)
boot_idx = rng_boot.choice(len(df_raw_train), B, replace=True)
df_boot = df_raw_train.iloc[boot_idx].reset_index(drop=True)
X_boot_encoded = encode_profiles(df_boot)
X_boot_std = ((X_boot_encoded - X_mean) / X_std).astype(np.float32)
y_boot = score_dnn(X_boot_encoded)

print(f'Bootstrap MC: {B} profiles resampled from training data')
print(f'  Predicted cost range: R{y_boot.min():,.2f} to R{y_boot.max():,.2f}')
print(f'  Mean: R{y_boot.mean():,.2f}')

# Compute SHAP on bootstrap sample
X_boot_fg = torch.tensor(X_boot_std, dtype=torch.float32)
shap_boot_list = []
for i in range(n_batches):
    start, end = i * BATCH_SHAP, min((i+1) * BATCH_SHAP, B)
    sv = explainer.shap_values(X_boot_fg[start:end])
    if isinstance(sv, list): sv = sv[0]
    if isinstance(sv, torch.Tensor): sv = sv.cpu().numpy()
    if sv.ndim == 3 and sv.shape[-1] == 1: sv = sv.squeeze(-1)
    shap_boot_list.append(sv)
    print(f'  Bootstrap SHAP batch {i+1}/{n_batches} done')

shap_boot_22 = np.concatenate(shap_boot_list, axis=0)

# Aggregate to 11 raw variables
shap_boot_11 = np.zeros((B, 11), dtype=np.float32)
for j, (var, cols) in enumerate(raw_to_encoded.items()):
    shap_boot_11[:, j] = shap_boot_22[:, cols].sum(axis=1)

# Compare
mc_mean_abs = np.abs(shap_values_11).mean(axis=0)
boot_mean_abs = np.abs(shap_boot_11).mean(axis=0)

df_invariance = pd.DataFrame({
    'Variable': RAW_VARS,
    'Independent MC |SHAP|': mc_mean_abs,
    'Bootstrap MC |SHAP|': boot_mean_abs,
    'Absolute Diff': np.abs(mc_mean_abs - boot_mean_abs),
    'Relative Diff (%)': np.abs(mc_mean_abs - boot_mean_abs) / boot_mean_abs * 100
})
df_invariance['Rank (Indep)'] = df_invariance['Independent MC |SHAP|'].rank(ascending=False).astype(int)
df_invariance['Rank (Boot)'] = df_invariance['Bootstrap MC |SHAP|'].rank(ascending=False).astype(int)
df_invariance = df_invariance.sort_values('Independent MC |SHAP|', ascending=False).reset_index(drop=True)

from scipy.stats import spearmanr
rho, p_val = spearmanr(mc_mean_abs, boot_mean_abs)

print(f'\nSHAP Invariance Test: Independent MC vs Bootstrap MC')
print(df_invariance.to_string(index=False, float_format='%.4f'))
print(f'\nSpearman rank correlation: rho = {rho:.3f} (p = {p_val:.4f})')
print(f'Max absolute SHAP difference: {df_invariance["Absolute Diff"].max():.4f}')
print(f'Max relative difference: {df_invariance["Relative Diff (%)"].max():.1f}%')

df_invariance.to_csv(FIG_DIR + 'shap_invariance_test.csv', index=False)

In [ ]:
# Figure: SHAP Invariance Test (3-panel: bar comparison, scatter + 45-deg line, rank comparison)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# (a) Side-by-side bar chart
df_inv_sorted = df_invariance.sort_values('Independent MC |SHAP|', ascending=True)
y_pos = np.arange(len(RAW_VARS))
w = 0.35
axes[0].barh(y_pos - w/2, [df_inv_sorted.loc[df_inv_sorted['Variable']==v, 'Independent MC |SHAP|'].values[0] for v in df_inv_sorted['Variable']],
             w, label='Independent MC', color='#2196F3', alpha=0.8)
axes[0].barh(y_pos + w/2, [df_inv_sorted.loc[df_inv_sorted['Variable']==v, 'Bootstrap MC |SHAP|'].values[0] for v in df_inv_sorted['Variable']],
             w, label='Bootstrap MC', color='#F44336', alpha=0.8)
axes[0].set_yticks(y_pos); axes[0].set_yticklabels(df_inv_sorted['Variable'])
axes[0].set_xlabel('Mean |SHAP|'); axes[0].set_title('(a) Feature Importance Comparison')
axes[0].legend(fontsize=9); sns.despine(ax=axes[0])

# (b) Scatter with 45-degree line
axes[1].scatter(boot_mean_abs, mc_mean_abs, s=60, color='#2196F3', edgecolors='black', linewidths=0.5, zorder=5)
for j, var in enumerate(RAW_VARS):
    axes[1].annotate(var, (boot_mean_abs[j], mc_mean_abs[j]), fontsize=7,
                     xytext=(5, 5), textcoords='offset points')
lims = [0, max(boot_mean_abs.max(), mc_mean_abs.max()) * 1.1]
axes[1].plot(lims, lims, 'r--', linewidth=1, alpha=0.7, label='45-degree line')
axes[1].set_xlabel('Bootstrap MC Mean |SHAP|'); axes[1].set_ylabel('Independent MC Mean |SHAP|')
axes[1].set_title(f'(b) SHAP Scatter ($\\rho$ = {rho:.3f})'); axes[1].legend(); sns.despine(ax=axes[1])

# (c) Rank comparison
ranks_indep = df_invariance.set_index('Variable')['Rank (Indep)']
ranks_boot = df_invariance.set_index('Variable')['Rank (Boot)']
rank_order = df_invariance.sort_values('Rank (Indep)')['Variable'].tolist()
y_r = np.arange(len(rank_order))
axes[2].barh(y_r - w/2, [ranks_indep[v] for v in rank_order], w, label='Independent MC', color='#2196F3', alpha=0.8)
axes[2].barh(y_r + w/2, [ranks_boot[v] for v in rank_order], w, label='Bootstrap MC', color='#F44336', alpha=0.8)
axes[2].set_yticks(y_r); axes[2].set_yticklabels(rank_order)
axes[2].set_xlabel('Rank'); axes[2].set_title('(c) Rank Comparison')
axes[2].invert_xaxis(); axes[2].legend(fontsize=9); sns.despine(ax=axes[2])

plt.suptitle('SHAP Invariance: Independent MC vs Bootstrap MC (Joint Distribution Test)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_shap_invariance_test.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: fig_shap_invariance_test.png')

## 11. Residual-Augmented MC Sensitivity

In [ ]:
y_train_pred = score_dnn(X_train_raw)
residuals_train = y_train_raw - y_train_pred
residuals_centred = residuals_train - residuals_train.mean()
sigma_e = residuals_centred.std()
print(f'Residuals: mean(centred)={residuals_centred.mean():.6f}, SD={sigma_e:.2f}')

rng_res = np.random.default_rng(SEED + 1)
eps_sample = rng_res.choice(residuals_centred, size=B, replace=True)
y_mc_augmented = y_mc + eps_sample
y_mc_upper = y_mc + 0.5 * sigma_e
y_mc_lower = y_mc - 0.5 * sigma_e

print(f'Augmented: Mean=R{y_mc_augmented.mean():,.2f} (base: R{y_mc.mean():,.2f})')
print(f'Augmented: SD=R{y_mc_augmented.std():,.2f} (base: R{y_mc.std():,.2f})')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(y_mc, bins=80, alpha=0.5, label='Base MC', color='#2196F3', density=True, edgecolor='white')
axes[0].hist(y_mc_augmented, bins=80, alpha=0.5, label='Residual-Augmented', color='#F44336', density=True, edgecolor='white')
axes[0].set_xlabel('Predicted Cost (R)'); axes[0].set_ylabel('Density')
axes[0].set_title('Base vs Residual-Augmented'); axes[0].legend(); sns.despine(ax=axes[0])
sort_idx = np.argsort(y_mc); step = max(1, B//2000); pi = np.arange(0, B, step)
axes[1].fill_between(pi, y_mc_lower[sort_idx][pi], y_mc_upper[sort_idx][pi], alpha=0.3, color='#F44336', label='$\\pm 0.5\\sigma_e$')
axes[1].plot(pi, y_mc[sort_idx][pi], color='#2196F3', linewidth=0.8, label='Base')
axes[1].set_xlabel('Profile Index (sorted)'); axes[1].set_ylabel('Cost (R)')
axes[1].set_title('Perturbation Envelope'); axes[1].legend(); sns.despine(ax=axes[1])
plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_mc_residual_augmented.png', dpi=300, bbox_inches='tight')
plt.show()

## 12. SHAP Global Attribution

In [ ]:
model_cpu = copy.deepcopy(final_model).cpu()
model_cpu.eval()

rng_shap = np.random.default_rng(SEED)
bg_idx = rng_shap.choice(len(X_train), 200, replace=False)
X_bg = torch.tensor(X_train[bg_idx], dtype=torch.float32)
X_fg = torch.tensor(X_mc_std, dtype=torch.float32)

explainer = shap.GradientExplainer(model_cpu, X_bg)

BATCH_SHAP = 2000
shap_values_list = []
n_batches = (B + BATCH_SHAP - 1) // BATCH_SHAP
for i in range(n_batches):
    start, end = i * BATCH_SHAP, min((i+1) * BATCH_SHAP, B)
    sv = explainer.shap_values(X_fg[start:end])
    if isinstance(sv, list): sv = sv[0]
    if isinstance(sv, torch.Tensor): sv = sv.cpu().numpy()
    if sv.ndim == 3 and sv.shape[-1] == 1: sv = sv.squeeze(-1)
    shap_values_list.append(sv)
    print(f'  Batch {i+1}/{n_batches} done')

shap_values_22 = np.concatenate(shap_values_list, axis=0)
print(f'SHAP values shape: {shap_values_22.shape}')

In [ ]:
# SHAP efficiency check
with torch.no_grad():
    phi_0 = model_cpu(X_bg).mean().item()
    y_mc_norm = model_cpu(X_fg).numpy().flatten()
reconstructed = phi_0 + shap_values_22.sum(axis=1)
eff_error = np.abs(reconstructed - y_mc_norm)
print(f'phi_0={phi_0:.6f}, Mean|error|={eff_error.mean():.6f}, Max|error|={eff_error.max():.6f}')

# Aggregate 22 encoded -> 11 raw variables
raw_to_encoded = {
    'age': [0], 'gender': [1], 'bmi': [2], 'children': [3], 'smoker': [4],
    'region': [5,6,7], 'medical_history': [8,9,10], 'family_medical_history': [11,12,13],
    'exercise_frequency': [14,15,16], 'occupation': [17,18,19], 'coverage_level': [20,21]
}
RAW_VARS = list(raw_to_encoded.keys())
shap_values_11 = np.zeros((B, 11), dtype=np.float32)
for j, (var, cols) in enumerate(raw_to_encoded.items()):
    shap_values_11[:, j] = shap_values_22[:, cols].sum(axis=1)

mean_abs = np.abs(shap_values_11).mean(axis=0)
mean_signed = shap_values_11.mean(axis=0)
df_global_shap = pd.DataFrame({'Variable': RAW_VARS, 'Mean Signed': mean_signed, 'Mean |SHAP|': mean_abs}
                              ).sort_values('Mean |SHAP|', ascending=False).reset_index(drop=True)
print('\nGlobal SHAP Importance:')
print(df_global_shap.to_string(index=False, float_format='%.6f'))

In [ ]:
# SHAP importance bar chart
df_plot = df_global_shap.sort_values('Mean |SHAP|', ascending=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(df_plot['Variable'], df_plot['Mean |SHAP|'], color='#2196F3', edgecolor='white')
axes[0].set_xlabel('Mean |SHAP|'); axes[0].set_title('Global Feature Importance'); sns.despine(ax=axes[0])
colours = ['#2196F3' if v >= 0 else '#F44336' for v in df_plot['Mean Signed']]
axes[1].barh(df_plot['Variable'], df_plot['Mean Signed'], color=colours, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Mean Signed SHAP'); axes[1].set_title('Global Feature Direction'); sns.despine(ax=axes[1])
plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_shap_global_importance.png', dpi=300, bbox_inches='tight')
plt.show()

# Beeswarm
shap_expl = shap.Explanation(values=shap_values_22, base_values=np.full(B, phi_0),
                              data=X_mc_std, feature_names=FEATURES)
fig, ax = plt.subplots(figsize=(10, 10))
shap.plots.beeswarm(shap_expl, max_display=22, show=False)
plt.title('SHAP Beeswarm (22 Encoded Features)'); plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_shap_beeswarm.png', dpi=300, bbox_inches='tight')
plt.show()

## 13. High-Cost Region Analysis and Conditional Shapley

In [ ]:
thresholds = {'tau_90': 0.90, 'tau_95': 0.95, 'tau_99': 0.99}
u_vals, h_sets = {}, {}
for name, tau in thresholds.items():
    u = np.percentile(y_mc, tau * 100)
    u_vals[name] = u
    h_sets[name] = y_mc >= u

print('High-Cost Thresholds:')
print(f'{"Threshold":<12} {"u (R)":>12} {"n(H)":>8} {"Proportion":>12}')
for name, tau in thresholds.items():
    n_h = h_sets[name].sum()
    print(f'{name:<12} {u_vals[name]:>12,.2f} {n_h:>8d} {n_h/B:>12.4f}')

# Profile high-cost subsets
for name in ['tau_90', 'tau_95', 'tau_99']:
    mask = h_sets[name]
    df_hc = df_mc[mask]
    print(f'\n--- {name} (n={mask.sum()}) ---')
    print(f'  smoker=yes: {(df_hc["smoker"]=="yes").mean():.2%}')
    print(f'  Premium:    {(df_hc["coverage_level"]=="Premium").mean():.2%}')

In [ ]:
# Conditional SHAP summaries
conditional_results = {}
for name in ['tau_90', 'tau_95', 'tau_99']:
    sv_cond = shap_values_11[h_sets[name]]
    df_cond = pd.DataFrame({'Variable': RAW_VARS,
        'Cond Mean Signed': sv_cond.mean(axis=0),
        'Cond Mean |SHAP|': np.abs(sv_cond).mean(axis=0)
    }).sort_values('Cond Mean |SHAP|', ascending=False).reset_index(drop=True)
    conditional_results[name] = df_cond
    print(f'\nConditional SHAP ({name}, n={h_sets[name].sum()}):')
    print(df_cond.to_string(index=False, float_format='%.6f'))

In [ ]:
# Conditional vs Global importance figure
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for idx, name in enumerate(['tau_90', 'tau_95', 'tau_99']):
    df_c = conditional_results[name].sort_values('Cond Mean |SHAP|', ascending=True)
    ax = axes[idx]
    ax.barh(df_c['Variable'], df_c['Cond Mean |SHAP|'], color='#F44336', edgecolor='white', alpha=0.7, label=f'Conditional ({name})')
    gv = [df_global_shap.loc[df_global_shap['Variable']==v, 'Mean |SHAP|'].values[0] for v in df_c['Variable']]
    ax.barh(df_c['Variable'], gv, color='#2196F3', edgecolor='white', alpha=0.4, label='Global')
    ax.set_xlabel('Mean |SHAP|'); ax.set_title(f'Conditional vs Global ({name})'); ax.legend(fontsize=8)
    sns.despine(ax=ax)
plt.suptitle('Conditional vs Global SHAP Importance', fontsize=14, y=1.02); plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_shap_conditional_importance.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP waterfall for representative high-cost profile
target_cost = u_vals['tau_95']
rep_idx = np.argmin(np.abs(y_mc - target_cost))
print(f'Representative profile (idx {rep_idx}): R{y_mc[rep_idx]:,.2f}')
for col in df_mc.columns:
    print(f'  {col}: {df_mc.iloc[rep_idx][col]}')

rep_expl = shap.Explanation(
    values=shap_values_11[rep_idx],
    base_values=phi_0,
    data=np.array([df_mc.iloc[rep_idx][v] for v in RAW_VARS], dtype=object),
    feature_names=RAW_VARS)
fig, ax = plt.subplots(figsize=(10, 8))
shap.plots.waterfall(rep_expl, max_display=11, show=False)
plt.title(f'SHAP Waterfall (R{y_mc[rep_idx]:,.0f})'); plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_shap_highcost_waterfall.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# High-cost profile comparison figure
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
compare_vars = [('age','c'), ('bmi','c'), ('children','c'), ('smoker','d'), ('coverage_level','d'), ('medical_history','d')]
colours_t = {'Full MC': '#2196F3', 'tau_90': '#4CAF50', 'tau_95': '#FF9800', 'tau_99': '#F44336'}
for idx, (var, vtype) in enumerate(compare_vars):
    ax = axes[idx//3, idx%3]
    if vtype == 'c':
        ax.hist(df_mc[var], bins=30, alpha=0.4, label='Full MC', color=colours_t['Full MC'], density=True, edgecolor='white')
        for nm in ['tau_90','tau_95','tau_99']:
            m = h_sets[nm]
            if m.sum() > 5:
                ax.hist(df_mc.loc[m, var], bins=20, alpha=0.4, label=nm, color=colours_t[nm], density=True, edgecolor='white')
        ax.set_xlabel(var); ax.set_ylabel('Density')
    else:
        cats = sorted(df_mc[var].unique()); x_pos = np.arange(len(cats)); w = 0.2
        offsets = {'Full MC': -1.5, 'tau_90': -0.5, 'tau_95': 0.5, 'tau_99': 1.5}
        for lbl, off in offsets.items():
            if lbl == 'Full MC': vals = [(df_mc[var]==c).mean() for c in cats]
            else:
                m = h_sets[lbl]
                vals = [(df_mc.loc[m, var]==c).mean() for c in cats] if m.sum() > 0 else [0]*len(cats)
            ax.bar(x_pos + off*w, vals, w, label=lbl, color=colours_t[lbl], alpha=0.7)
        ax.set_xticks(x_pos); ax.set_xticklabels(cats, rotation=45, ha='right', fontsize=8); ax.set_ylabel('Proportion')
    ax.set_title(var.replace('_',' ').title()); ax.legend(fontsize=7); sns.despine(ax=ax)
plt.suptitle('High-Cost Profile Comparison', fontsize=14, y=1.01); plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_highcost_profile_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Threshold comparison grouped bar
fig, ax = plt.subplots(figsize=(12, 7))
global_order = df_global_shap.sort_values('Mean |SHAP|', ascending=False)['Variable'].tolist()
x_pos = np.arange(len(RAW_VARS)); w = 0.2
gv = [df_global_shap.loc[df_global_shap['Variable']==v, 'Mean |SHAP|'].values[0] for v in global_order]
ax.bar(x_pos - 1.5*w, gv, w, label='Global', color='#2196F3', alpha=0.8)
cond_colours = {'tau_90': '#4CAF50', 'tau_95': '#FF9800', 'tau_99': '#F44336'}
for i, nm in enumerate(['tau_90','tau_95','tau_99']):
    df_c = conditional_results[nm]
    vals = [df_c.loc[df_c['Variable']==v, 'Cond Mean |SHAP|'].values[0] for v in global_order]
    ax.bar(x_pos + (i-0.5)*w, vals, w, label=nm, color=cond_colours[nm], alpha=0.7)
ax.set_xticks(x_pos); ax.set_xticklabels(global_order, rotation=45, ha='right')
ax.set_ylabel('Mean |SHAP|'); ax.set_title('Feature Importance: Global vs Conditional'); ax.legend()
sns.despine(ax=ax); plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_shap_threshold_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 14. Summary Tables and Exports

In [ ]:
# MC summary
pd.DataFrame([mc_summary]).to_csv(FIG_DIR + 'mc_summary.csv', index=False)
df_conv.to_csv(FIG_DIR + 'mc_convergence.csv', index=False)

# SHAP importance table
shap_table = df_global_shap[['Variable', 'Mean Signed', 'Mean |SHAP|']].copy()
for nm in ['tau_90','tau_95','tau_99']:
    dc = conditional_results[nm]
    shap_table = shap_table.merge(dc[['Variable','Cond Mean Signed','Cond Mean |SHAP|']].rename(
        columns={'Cond Mean Signed': f'Signed ({nm})', 'Cond Mean |SHAP|': f'|SHAP| ({nm})'}),
        on='Variable', how='left')
shap_table.to_csv(FIG_DIR + 'shap_importance_table.csv', index=False)

# High-cost thresholds
hc_rows = [{'Threshold': nm, 'Tau': tau, 'u (R)': u_vals[nm], 'n(H)': int(h_sets[nm].sum()),
            'Proportion': h_sets[nm].sum()/B} for nm, tau in thresholds.items()]
pd.DataFrame(hc_rows).to_csv(FIG_DIR + 'highcost_thresholds.csv', index=False)

# Full export
df_export = df_mc.copy()
df_export['predicted_cost'] = y_mc
df_export['predicted_cost_augmented'] = y_mc_augmented
for j, var in enumerate(RAW_VARS):
    df_export[f'shap_{var}'] = shap_values_11[:, j]
df_export.to_csv(FIG_DIR + 'mc_profiles_with_shap.csv', index=False)

print('All CSV exports saved to:', FIG_DIR)

In [ ]:
# Final summary
print('=' * 70)
print('CHAPTER 3 COMPLETE (60/20/20 SPLIT, NO SamplingForML)')
print('=' * 70)
print(f'\nArchitecture: 22 -> 256 -> 128 -> 64 -> 32 -> 16 -> 1')
total_params = sum(p.numel() for p in final_model.parameters())
print(f'Parameters  : {total_params:,}')
print()
print('Best Hyperparameters:')
print(f'  Activation   : {best_activation}')
print(f'  Learning rate: {best_lr}')
print(f'  Batch size   : {best_bs}')
print(f'  Optimiser    : {best_optimiser_name}')
print(f'  Momentum     : B1={best_betas[0]}, B2={best_betas[1]}')
print(f'  Best epoch   : {final_result["best_epoch"]}')
print()
print('DNN Performance (de-normalised, Rand scale):')
print(f'{"Set":<12} {"R2":>8} {"RMSE (R)":>12} {"MAE (R)":>12}')
print(f'{"-"*44}')
for set_name, m in metrics.items():
    print(f'{set_name:<12} {m["R2"]:>8.4f} {m["RMSE"]:>12.2f} {m["MAE"]:>12.2f}')
